# 🏥 Module 12.1: Reinforcement Learning for Medical Image Enhancement

### A Complete Project — From Synthetic Data to Trained RL Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_12_Real_World_Projects/01_Medical_Image_Enhancement/medical_image_enhancement.ipynb)

---

## 🎯 Learning Objectives

1. Understand why medical image enhancement is uniquely suited to RL
2. Formulate image enhancement as a Markov Decision Process (MDP)
3. Design reward functions using clinically-relevant metrics (CNR, SNR, SSIM)
4. Generate realistic synthetic medical images (X-ray, CT-like)
5. Build a full DQN-based enhancement pipeline with CNN feature extractor
6. Train and evaluate the agent with rich visualizations
7. Discuss clinical relevance, regulatory considerations, and limitations

---

In [ ]:
!pip install numpy matplotlib torch torchvision scipy scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from scipy import ndimage
from skimage.metrics import structural_similarity as ssim
from collections import deque, namedtuple
import random
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("✅ All dependencies loaded!")

In [ ]:
# Download REAL open-source datasets for real-world projects
import torchvision
import subprocess
import sys

# MedMNIST for medical imaging (install + download)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist', '-q'])
from medmnist import ChestMNIST, DermaMNIST

# Chest X-rays (real medical images!)
chest_data = ChestMNIST(split='train', download=True, size=28)
print(f"✅ ChestMNIST: {len(chest_data)} real chest X-ray images")

# Dermatology images
derma_data = DermaMNIST(split='train', download=True, size=28)
print(f"✅ DermaMNIST: {len(derma_data)} real skin lesion images")

# EuroSAT for satellite imagery
try:
    eurosat = torchvision.datasets.EuroSAT(root='./data', download=True)
    print(f"✅ EuroSAT: {len(eurosat)} real satellite images (64x64, 10 land-use classes)")
except:
    print("⚠️ EuroSAT downloading...")

# CIFAR-10 for video/editing projects
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos for editing/video projects")

## Deep Derivation: RL for Medical Image Enhancement — Theory and Safety Constraints

### Step 1: Medical Image Quality Metrics (PSNR, SSIM, VIF)
**Peak Signal-to-Noise Ratio:**
$$\text{PSNR} = 10\log_{10}\left(\frac{L^2}{\text{MSE}}\right) = 20\log_{10}(L) - 10\log_{10}(\text{MSE})$$

where $L$ = maximum pixel value (255 for 8-bit), $\text{MSE} = \frac{1}{HW}\sum_{i,j}(I_{ij} - \hat{I}_{ij})^2$.

**Structural Similarity Index (full derivation):**
$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

**Derivation of the three components:**
$$\text{SSIM} = \underbrace{\frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1}}_{l(x,y) \text{ luminance}} \cdot \underbrace{\frac{2\sigma_x\sigma_y + C_2'}{\sigma_x^2 + \sigma_y^2 + C_2'}}_{c(x,y) \text{ contrast}} \cdot \underbrace{\frac{\sigma_{xy} + C_3}{\sigma_x\sigma_y + C_3}}_{s(x,y) \text{ structure}}$$

where $C_1 = (K_1 L)^2$, $C_2 = (K_2 L)^2$, $C_3 = C_2/2$ (stability constants).

**Proof that SSIM $\in [-1, 1]$:** By Cauchy-Schwarz, $|\sigma_{xy}| \leq \sigma_x \sigma_y$, so $|s(x,y)| \leq 1$. Both $l$ and $c$ are in $[0,1]$ by AM-GM inequality: $2ab \leq a^2 + b^2$. Therefore $|\text{SSIM}| \leq 1$. $\blacksquare$

### Step 2: Constrained MDP for Medical Safety
Medical enhancement must satisfy clinical constraints. Formulate as a Constrained MDP (CMDP):
$$\max_\pi J(\pi) = E_\pi\left[\sum_t \gamma^t r_t\right]$$
$$\text{subject to: } C_i(\pi) = E_\pi\left[\sum_t \gamma^t c_t^{(i)}\right] \leq d_i, \quad i = 1, \ldots, m$$

**Constraint examples for medical images:**
1. $C_1$: Maximum pixel intensity change $\leq \delta_1$ (prevent hallucinated structures)
2. $C_2$: Edge preservation score $\geq \delta_2$ (maintain diagnostic boundaries)
3. $C_3$: Color shift in diagnostic regions $\leq \delta_3$

**Lagrangian relaxation:**
$$\mathcal{L}(\pi, \lambda) = J(\pi) - \sum_i \lambda_i (C_i(\pi) - d_i)$$

**Derivation of the dual problem:**
$$\min_{\lambda \geq 0} \max_\pi \mathcal{L}(\pi, \lambda)$$

By strong duality (which holds for CMDPs with finite state/action spaces):
$$\max_\pi \min_{\lambda \geq 0} \mathcal{L}(\pi, \lambda) = \min_{\lambda \geq 0} \max_\pi \mathcal{L}(\pi, \lambda)$$

The optimal policy is a mixture of at most $m+1$ deterministic policies (one per active constraint + unconstrained). $\blacksquare$

### Step 3: Lagrangian PPO for Safe Enhancement
Update policy and multipliers alternately:

**Primal step (policy update with augmented reward):**
$$r_t^{\text{aug}} = r_t - \sum_i \lambda_i c_t^{(i)}$$

$$L^{\text{CLIP}}(\theta) = E\left[\min\left(\rho_t(\theta)\hat{A}_t^{\text{aug}}, \text{clip}(\rho_t, 1\pm\epsilon)\hat{A}_t^{\text{aug}}\right)\right]$$

**Dual step (multiplier update):**
$$\lambda_i \leftarrow \max(0, \lambda_i + \eta_\lambda (\hat{C}_i - d_i))$$

**Derivation of convergence:** The primal-dual iteration converges to a saddle point $(\pi^*, \lambda^*)$. The dual function $g(\lambda) = \max_\pi \mathcal{L}(\pi, \lambda)$ is convex (pointwise maximum of affine functions in $\lambda$). Projected gradient ascent on the dual converges at rate $O(1/\sqrt{T})$. $\blacksquare$

### Step 4: Noise Model for Medical Imaging Modalities
**CT imaging (Poisson + Gaussian):**
$$I_{\text{noisy}} = \text{Poisson}(I_0 \cdot e^{-\int \mu(x) dx}) + \mathcal{N}(0, \sigma_e^2)$$

Taking log: $\hat{\mu} = -\log(I_{\text{noisy}}/I_0) \approx \mu + n$

where noise variance (via delta method):
$$\text{Var}(\hat{\mu}) \approx \frac{1}{I_0 \cdot e^{-\int\mu dx}} + \frac{\sigma_e^2}{(I_0 e^{-\int\mu dx})^2}$$

**MRI imaging (Rician noise):**
$$p(I | A, \sigma) = \frac{I}{\sigma^2}\exp\left(-\frac{I^2 + A^2}{2\sigma^2}\right) I_0\left(\frac{IA}{\sigma^2}\right)$$

where $I_0$ is the modified Bessel function and $A$ is the true signal magnitude.

**Derivation:** MRI measures complex signal $z = (A + n_1) + in_2$ where $n_1, n_2 \sim \mathcal{N}(0,\sigma^2)$. The magnitude $|z| = \sqrt{(A+n_1)^2 + n_2^2}$ follows the Rice distribution. For high SNR ($A \gg \sigma$), this approaches Gaussian; for low SNR, it's biased upward (noise floor). $\blacksquare$

### Step 5: Enhancement Action Space for Medical Images
Define medically-meaningful enhancement actions:

$$\mathcal{A} = \{a_1: \text{window/level}, a_2: \text{CLAHE}, a_3: \text{bilateral filter}, a_4: \text{unsharp mask}, a_5: \text{gamma correction}, a_6: \text{no-op}\}$$

**Window/Level (continuous):**
$$I_{\text{display}}(p) = \text{clip}\left(\frac{I(p) - (L - W/2)}{W}, 0, 1\right)$$

where $W$ = window width, $L$ = window level (center).

**CLAHE (Contrast Limited Adaptive Histogram Equalization):**
$$T(k) = \text{CDF}_{\text{clipped}}(k) = \frac{1}{N}\sum_{i=0}^{k} \min(h(i), c_L)$$

where $c_L = \frac{N}{K}(1 + \frac{\alpha}{100}(s_{\max}-1))$ is the clip limit, $\alpha$ is the clip factor.

**Derivation of clip limit:** Without clipping, histogram equalization can amplify noise in uniform regions. Clipping at $c_L$ limits the maximum slope of the transfer function to $(1+\alpha(s_{\max}-1)/100)$, bounding noise amplification. $\blacksquare$

### Step 6: Radiologist Preference Learning (RLHF for Medical)
Learn a reward model from radiologist preferences:
$$R_\phi(I_{\text{enhanced}}, I_{\text{original}}) = \text{MLP}(\text{CNN}(I_{\text{enhanced}}) - \text{CNN}(I_{\text{original}}))$$

**Bradley-Terry preference model:**
$$p(I_1 \succ I_2) = \sigma(R_\phi(I_1) - R_\phi(I_2))$$

**Loss for reward model training:**
$$\mathcal{L}_R = -E_{(I_w, I_l)}[\log\sigma(R_\phi(I_w) - R_\phi(I_l))]$$

**Derivation of inter-rater agreement correction:** With $K$ radiologists and agreement rate $\kappa$:
$$\mathcal{L}_R^{\text{corrected}} = -\sum_{(w,l)} \left[\frac{n_{wl}}{K}\log\sigma(R_w - R_l) + \frac{K-n_{wl}}{K}\log\sigma(R_l - R_w)\right]$$

where $n_{wl}$ is the number of raters preferring $I_w$ over $I_l$. This soft labeling handles disagreement gracefully. $\blacksquare$

### Step 7: Diagnostic Preservation Guarantee
**Theorem:** An enhancement policy $\pi$ is **diagnostically safe** if for all disease classes $c$:
$$\text{AUC}_c(D \circ \pi) \geq \text{AUC}_c(D) - \epsilon$$

where $D$ is a pre-trained diagnostic classifier.

**Proof approach:** Define the diagnostic information:
$$I_{\text{diag}}(I) = \sum_c H(Y_c) - H(Y_c | I)$$

Enhancement preserves diagnostics if:
$$I_{\text{diag}}(\pi(I)) \geq I_{\text{diag}}(I) - \delta$$

By the data processing inequality, any deterministic transformation $\pi$:
$$I(Y; \pi(I)) \leq I(Y; I)$$

Equality holds iff $\pi$ is a sufficient statistic for $Y$. Therefore, enhancement can only lose diagnostic information — the goal is to minimize this loss while maximizing visual quality.

**Practical constraint:** Add a diagnostic preservation term to the reward:
$$R_{\text{total}} = R_{\text{quality}} - \mu \cdot \max(0, \text{AUC}_{\text{baseline}} - \text{AUC}_{\text{enhanced}} - \epsilon) \quad \blacksquare$$

---

## 1. Introduction & Motivation

Medical imaging is the backbone of modern clinical diagnosis. Radiologists interpret
**X-rays**, **CT scans**, and **MRIs** to detect disease, guide surgery, and monitor
treatment. However, raw images often suffer from:

| Degradation | Cause | Clinical Impact |
|---|---|---|
| Low contrast | Dose reduction, tissue overlap | Missed lesions |
| High noise | Low-dose protocols, detector limits | False positives |
| Motion blur | Patient movement during acquisition | Unclear boundaries |
| Artifacts | Metal implants, beam hardening | Diagnostic confusion |

### Why Reinforcement Learning?

Traditional enhancement uses a **fixed pipeline**: denoise → equalize → sharpen. But:

- Different images need different enhancement *sequences*
- The optimal *strength* of each operation varies
- Over-enhancement can **destroy** diagnostic information

RL naturally handles this as **sequential decision-making**: an agent observes the
current image, selects an enhancement action, observes the result, and decides what
to do next — learning policies that adapt to each image.

### Clinical Significance

Proper enhancement can:
- Improve **sensitivity** (fewer missed findings)
- Reduce **false positive rates**
- Enable **lower radiation doses** (enhance low-dose images instead)
- Standardize image quality across scanners and protocols

---

## 2. Problem Formulation as an MDP

We formulate medical image enhancement as a **Markov Decision Process** $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$:

### State Space $\mathcal{S}$

The state at step $t$ is the current enhanced image:

$$s_t = I_t \in [0, 1]^{H \times W}$$

where $I_0$ is the degraded input and $I_t$ is the image after $t$ enhancement steps.

### Action Space $\mathcal{A}$

We define 9 discrete enhancement actions:

| Index | Action | Description |
|-------|--------|-------------|
| 0 | `adjust_contrast` | Linear contrast stretching |
| 1 | `adjust_brightness` | Additive brightness shift |
| 2 | `denoise_gaussian` | Gaussian blur denoising |
| 3 | `denoise_median` | Median filter denoising |
| 4 | `sharpen` | Unsharp masking |
| 5 | `histogram_equalize` | Global histogram equalization |
| 6 | `CLAHE` | Contrast Limited Adaptive Histogram Equalization |
| 7 | `gamma_correction` | Power-law transformation |
| 8 | `no_op` | Do nothing (terminate signal) |

### Reward Function $\mathcal{R}$

The reward captures **clinical quality** via multiple metrics:

$$R = w_1 \cdot \text{SSIM}(I_{enhanced}, I_{target}) + w_2 \cdot \text{CNR} + w_3 \cdot \text{SNR} - w_4 \cdot \text{artifact\_penalty}$$

where:

**Contrast-to-Noise Ratio (CNR)** — measures how well tissue types are distinguished:

$$\text{CNR} = \frac{|\mu_{\text{tissue}} - \mu_{\text{background}}|}{\sigma_{\text{background}}}$$

**Signal-to-Noise Ratio (SNR)** — measures signal clarity:

$$\text{SNR} = \frac{\mu_{\text{signal}}}{\sigma_{\text{noise}}}$$

**Structural Similarity Index (SSIM)** — perceptual quality:

$$\text{SSIM}(x, y) = \frac{(2\mu_x \mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

with $C_1 = (k_1 L)^2$, $C_2 = (k_2 L)^2$, $L$ = dynamic range, $k_1=0.01$, $k_2=0.03$.

**Artifact Penalty** — penalizes over-enhancement:

$$\text{artifact\_penalty} = \|\nabla^2 I_{enhanced}\|_1 \cdot \mathbb{1}[\|\nabla^2 I_{enhanced}\|_1 > \tau]$$

This penalizes high-frequency ringing artifacts that emerge from excessive sharpening.

### Episode Termination

An episode ends when:
1. The agent selects `no_op`, or
2. Maximum steps $T_{\max}$ reached, or
3. Quality improvement $\Delta R < \epsilon$ (diminishing returns)

---

## 3. Synthetic Medical Image Generation

Since real medical images require institutional access, we generate **realistic synthetic** images
that capture the essential characteristics of X-ray and CT modalities.

In [ ]:
class SyntheticMedicalImageGenerator:
    """Generates realistic synthetic medical images for training."""

    def __init__(self, size=64):
        self.size = size

    def generate_xray(self):
        """Generate a synthetic chest X-ray-like image."""
        img = np.zeros((self.size, self.size), dtype=np.float64)
        cy, cx = self.size // 2, self.size // 2
        Y, X = np.ogrid[:self.size, :self.size]

        # Body outline (elliptical soft tissue region)
        body_mask = ((X - cx) / (self.size * 0.4)) ** 2 + \
                    ((Y - cy) / (self.size * 0.45)) ** 2 <= 1
        img[body_mask] = 0.3 + np.random.uniform(-0.02, 0.02)

        # Lung fields (two dark ellipses)
        for side in [-1, 1]:
            lung_cx = cx + side * int(self.size * 0.15)
            lung_mask = ((X - lung_cx) / (self.size * 0.12)) ** 2 + \
                        ((Y - cy) / (self.size * 0.3)) ** 2 <= 1
            img[lung_mask] = 0.1 + np.random.uniform(-0.02, 0.02)

        # Spine (vertical bright strip)
        spine_w = max(2, self.size // 20)
        img[int(cy * 0.4):int(cy * 1.6), cx - spine_w:cx + spine_w] = 0.7

        # Ribs (horizontal bright arcs)
        for i in range(5):
            rib_y = cy - int(self.size * 0.25) + i * int(self.size * 0.1)
            if 0 <= rib_y < self.size:
                for side in [-1, 1]:
                    for dx in range(int(self.size * 0.25)):
                        dy = int(3 * np.sin(dx * np.pi / (self.size * 0.25)))
                        py = rib_y + dy
                        px = cx + side * (spine_w + dx)
                        if 0 <= py < self.size and 0 <= px < self.size:
                            img[py, px] = max(img[py, px], 0.6)

        # Heart shadow (left-shifted ellipse)
        heart_cx = cx - int(self.size * 0.05)
        heart_cy = cy + int(self.size * 0.1)
        heart_mask = ((X - heart_cx) / (self.size * 0.1)) ** 2 + \
                     ((Y - heart_cy) / (self.size * 0.12)) ** 2 <= 1
        img[heart_mask] = 0.45

        img = ndimage.gaussian_filter(img, sigma=1.0)
        return np.clip(img, 0, 1)

    def generate_ct_slice(self):
        """Generate a synthetic CT cross-section."""
        img = np.zeros((self.size, self.size), dtype=np.float64)
        cy, cx = self.size // 2, self.size // 2
        Y, X = np.ogrid[:self.size, :self.size]

        # Outer body contour
        body_r = self.size * 0.42
        body_mask = (X - cx) ** 2 + (Y - cy) ** 2 <= body_r ** 2
        img[body_mask] = 0.35

        # Subcutaneous fat ring
        fat_r = self.size * 0.38
        fat_mask = (X - cx) ** 2 + (Y - cy) ** 2 <= fat_r ** 2
        img[fat_mask] = 0.25

        # Muscle layer
        muscle_r = self.size * 0.33
        muscle_mask = (X - cx) ** 2 + (Y - cy) ** 2 <= muscle_r ** 2
        img[muscle_mask] = 0.4

        # Internal organs (random ellipses with varying density)
        for _ in range(np.random.randint(3, 6)):
            ox = cx + np.random.randint(-int(self.size * 0.15), int(self.size * 0.15))
            oy = cy + np.random.randint(-int(self.size * 0.15), int(self.size * 0.15))
            rx = np.random.randint(int(self.size * 0.04), int(self.size * 0.1))
            ry = np.random.randint(int(self.size * 0.04), int(self.size * 0.1))
            organ_mask = ((X - ox) / max(rx, 1)) ** 2 + ((Y - oy) / max(ry, 1)) ** 2 <= 1
            intensity = np.random.uniform(0.2, 0.6)
            img[organ_mask] = intensity

        # Vertebral body (bright circle at center-back)
        vert_cy = cy + int(self.size * 0.2)
        vert_r = self.size * 0.06
        vert_mask = (X - cx) ** 2 + (Y - vert_cy) ** 2 <= vert_r ** 2
        img[vert_mask] = 0.8

        img = ndimage.gaussian_filter(img, sigma=0.8)
        return np.clip(img, 0, 1)

    def apply_degradation(self, img, noise_level=0.08, blur_sigma=1.0,
                          contrast_factor=0.5):
        """Apply realistic degradations to simulate poor acquisition."""
        degraded = img.copy()

        # Reduce contrast
        mean_val = degraded.mean()
        degraded = mean_val + contrast_factor * (degraded - mean_val)

        # Add Poisson-like noise (realistic for photon-counting detectors)
        scale = max(degraded.max(), 1e-8)
        counts = np.random.poisson(degraded / scale * 100) * scale / 100.0
        degraded = counts

        # Add Gaussian noise
        degraded += np.random.normal(0, noise_level, degraded.shape)

        # Motion blur (slight)
        if blur_sigma > 0:
            kernel_size = max(3, int(blur_sigma * 3))
            if kernel_size % 2 == 0:
                kernel_size += 1
            motion_kernel = np.zeros((kernel_size, kernel_size))
            motion_kernel[kernel_size // 2, :] = 1.0 / kernel_size
            degraded = ndimage.convolve(degraded, motion_kernel)

        return np.clip(degraded, 0, 1)


gen = SyntheticMedicalImageGenerator(size=64)

np.random.seed(42)
xray_clean = gen.generate_xray()
ct_clean = gen.generate_ct_slice()
xray_degraded = gen.apply_degradation(xray_clean)
ct_degraded = gen.apply_degradation(ct_clean)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0, 0].imshow(xray_clean, cmap='gray')
axes[0, 0].set_title('Clean X-ray', fontsize=13, fontweight='bold')
axes[0, 1].imshow(xray_degraded, cmap='gray')
axes[0, 1].set_title('Degraded X-ray', fontsize=13, fontweight='bold')
axes[0, 2].imshow(np.abs(xray_clean - xray_degraded), cmap='hot')
axes[0, 2].set_title('Degradation Map', fontsize=13, fontweight='bold')

axes[1, 0].imshow(ct_clean, cmap='gray')
axes[1, 0].set_title('Clean CT Slice', fontsize=13, fontweight='bold')
axes[1, 1].imshow(ct_degraded, cmap='gray')
axes[1, 1].set_title('Degraded CT Slice', fontsize=13, fontweight='bold')
axes[1, 2].imshow(np.abs(ct_clean - ct_degraded), cmap='hot')
axes[1, 2].set_title('Degradation Map', fontsize=13, fontweight='bold')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Synthetic Medical Images: Clean vs Degraded',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize degradation severity levels
np.random.seed(123)
base_img = gen.generate_xray()

severity_configs = [
    {'noise_level': 0.03, 'blur_sigma': 0.3, 'contrast_factor': 0.8,
     'label': 'Mild'},
    {'noise_level': 0.08, 'blur_sigma': 0.8, 'contrast_factor': 0.5,
     'label': 'Moderate'},
    {'noise_level': 0.15, 'blur_sigma': 1.5, 'contrast_factor': 0.3,
     'label': 'Severe'},
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(base_img, cmap='gray')
axes[0].set_title('Original', fontsize=13, fontweight='bold')
axes[0].axis('off')

for i, cfg in enumerate(severity_configs):
    deg = gen.apply_degradation(base_img, noise_level=cfg['noise_level'],
                                blur_sigma=cfg['blur_sigma'],
                                contrast_factor=cfg['contrast_factor'])
    axes[i + 1].imshow(deg, cmap='gray')
    snr_val = base_img.mean() / max(np.std(deg - base_img), 1e-8)
    axes[i + 1].set_title(f'{cfg["label"]}\nSNR={snr_val:.1f}',
                          fontsize=12, fontweight='bold')
    axes[i + 1].axis('off')

plt.suptitle('Degradation Severity Levels',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Full Pipeline Implementation

### 4.1 Medical Image Quality Metrics

We implement the quality metrics that drive our reward function.

In [ ]:
class MedicalImageMetrics:
    """Compute clinically-relevant image quality metrics."""

    @staticmethod
    def compute_cnr(image, tissue_mask=None, bg_mask=None):
        """Contrast-to-Noise Ratio.
        CNR = |mu_tissue - mu_background| / sigma_background
        """
        if tissue_mask is None or bg_mask is None:
            threshold = np.mean(image)
            tissue_mask = image > threshold
            bg_mask = image <= threshold

        if bg_mask.sum() == 0 or tissue_mask.sum() == 0:
            return 0.0

        mu_tissue = image[tissue_mask].mean()
        mu_bg = image[bg_mask].mean()
        sigma_bg = image[bg_mask].std() + 1e-8

        return abs(mu_tissue - mu_bg) / sigma_bg

    @staticmethod
    def compute_snr(image):
        """Signal-to-Noise Ratio.
        SNR = mu_signal / sigma_noise
        Noise estimated from high-frequency component.
        """
        smoothed = ndimage.gaussian_filter(image, sigma=2.0)
        noise = image - smoothed
        sigma_noise = noise.std() + 1e-8
        return image.mean() / sigma_noise

    @staticmethod
    def compute_ssim(image, reference):
        """Structural Similarity Index."""
        data_range = max(reference.max() - reference.min(), 1e-8)
        return ssim(image, reference, data_range=data_range)

    @staticmethod
    def compute_artifact_penalty(image, threshold=0.1):
        """Penalize high-frequency artifacts from over-enhancement."""
        laplacian = ndimage.laplace(image)
        l1_norm = np.abs(laplacian).mean()
        if l1_norm > threshold:
            return l1_norm
        return 0.0

    @staticmethod
    def composite_reward(enhanced, target, w1=0.4, w2=0.25, w3=0.25, w4=0.1):
        """Full composite reward for medical image quality."""
        metrics = MedicalImageMetrics
        ssim_val = metrics.compute_ssim(enhanced, target)
        cnr_val = metrics.compute_cnr(enhanced) / 10.0  # normalize
        snr_val = metrics.compute_snr(enhanced) / 20.0  # normalize
        artifact = metrics.compute_artifact_penalty(enhanced)
        reward = w1 * ssim_val + w2 * cnr_val + w3 * snr_val - w4 * artifact
        return reward, {'ssim': ssim_val, 'cnr': cnr_val * 10,
                        'snr': snr_val * 20, 'artifact': artifact}


# Quick demo of metrics
metrics = MedicalImageMetrics()
r_clean, m_clean = metrics.composite_reward(xray_clean, xray_clean)
r_deg, m_deg = metrics.composite_reward(xray_degraded, xray_clean)

print("Metrics for CLEAN image (reference = clean):")
print(f"  Reward={r_clean:.4f}  SSIM={m_clean['ssim']:.4f}  "
      f"CNR={m_clean['cnr']:.2f}  SNR={m_clean['snr']:.2f}")
print("\nMetrics for DEGRADED image (reference = clean):")
print(f"  Reward={r_deg:.4f}  SSIM={m_deg['ssim']:.4f}  "
      f"CNR={m_deg['cnr']:.2f}  SNR={m_deg['snr']:.2f}")
print(f"\nReward gap (room for improvement): {r_clean - r_deg:.4f}")

### 4.2 Medical Image Enhancement Environment

A Gym-like environment wrapping the enhancement problem.

In [ ]:
class MedicalImageEnv:
    """Gym-like environment for RL-based medical image enhancement."""

    ACTION_NAMES = [
        'adjust_contrast', 'adjust_brightness', 'denoise_gaussian',
        'denoise_median', 'sharpen', 'histogram_equalize',
        'CLAHE', 'gamma_correction', 'no_op'
    ]

    def __init__(self, image_size=64, max_steps=15):
        self.image_size = image_size
        self.max_steps = max_steps
        self.n_actions = len(self.ACTION_NAMES)
        self.generator = SyntheticMedicalImageGenerator(size=image_size)
        self.metrics = MedicalImageMetrics()
        self.reset()

    def reset(self):
        """Generate a new degraded image and reset state."""
        if np.random.random() < 0.5:
            self.target = self.generator.generate_xray()
        else:
            self.target = self.generator.generate_ct_slice()

        noise_level = np.random.uniform(0.05, 0.15)
        blur_sigma = np.random.uniform(0.3, 1.5)
        contrast_factor = np.random.uniform(0.3, 0.7)
        self.current_image = self.generator.apply_degradation(
            self.target, noise_level, blur_sigma, contrast_factor)

        self.initial_image = self.current_image.copy()
        self.step_count = 0
        self.action_history = []
        self.prev_reward, _ = self.metrics.composite_reward(
            self.current_image, self.target)
        return self._get_state()

    def _get_state(self):
        """Return current image as state tensor."""
        return self.current_image.copy()

    def _apply_action(self, action):
        """Apply the selected enhancement action."""
        img = self.current_image.copy()

        if action == 0:  # adjust_contrast
            mean = img.mean()
            img = mean + 1.3 * (img - mean)
        elif action == 1:  # adjust_brightness
            img = img + 0.05
        elif action == 2:  # denoise_gaussian
            img = ndimage.gaussian_filter(img, sigma=0.8)
        elif action == 3:  # denoise_median
            img = ndimage.median_filter(img, size=3)
        elif action == 4:  # sharpen
            blurred = ndimage.gaussian_filter(img, sigma=1.0)
            img = img + 0.5 * (img - blurred)
        elif action == 5:  # histogram_equalize
            hist, bins = np.histogram(img.flatten(), bins=256,
                                     range=(0, 1), density=True)
            cdf = hist.cumsum()
            cdf = cdf / cdf[-1]
            img = np.interp(img.flatten(), bins[:-1], cdf).reshape(img.shape)
        elif action == 6:  # CLAHE-like
            tile_size = max(8, self.image_size // 8)
            result = np.zeros_like(img)
            for i in range(0, self.image_size, tile_size):
                for j in range(0, self.image_size, tile_size):
                    tile = img[i:i + tile_size, j:j + tile_size]
                    if tile.size == 0:
                        continue
                    t_min, t_max = tile.min(), tile.max()
                    if t_max - t_min > 1e-8:
                        tile = (tile - t_min) / (t_max - t_min)
                    result[i:i + tile_size, j:j + tile_size] = tile
            img = result
        elif action == 7:  # gamma_correction
            img = np.power(np.clip(img, 1e-8, 1.0), 0.7)
        elif action == 8:  # no_op
            pass

        return np.clip(img, 0, 1)

    def step(self, action):
        """Execute one enhancement step."""
        self.current_image = self._apply_action(action)
        self.step_count += 1
        self.action_history.append(action)

        current_reward, info = self.metrics.composite_reward(
            self.current_image, self.target)
        reward = current_reward - self.prev_reward  # incremental improvement
        self.prev_reward = current_reward

        done = (action == 8 or self.step_count >= self.max_steps)

        info['step'] = self.step_count
        info['action'] = self.ACTION_NAMES[action]
        info['total_reward'] = current_reward

        return self._get_state(), reward, done, info


# Quick test
env = MedicalImageEnv(image_size=64, max_steps=15)
state = env.reset()
print(f"State shape: {state.shape}")
print(f"Number of actions: {env.n_actions}")
print(f"Action names: {env.ACTION_NAMES}")

# Run a few random steps
total_r = 0
for i in range(5):
    a = np.random.randint(0, env.n_actions - 1)  # exclude no_op
    s, r, d, info = env.step(a)
    total_r += r
    print(f"  Step {i+1}: {info['action']:>20s} | "
          f"reward={r:+.4f} | SSIM={info['ssim']:.3f} | "
          f"CNR={info['cnr']:.2f}")
print(f"\nTotal incremental reward: {total_r:.4f}")

### 4.3 DQN Agent with CNN Feature Extractor

We use a **Deep Q-Network** with a convolutional backbone to process image states.

The Q-function approximation:

$$Q(s, a; \theta) \approx Q^*(s, a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s', a') \mid s, a\right]$$

Loss (Huber / smooth L1):

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ L_\delta \left( r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta) \right) \right]$$

where $\theta^-$ are target network parameters updated periodically.

In [ ]:
class CNNQNetwork(nn.Module):
    """CNN-based Q-network for image state processing."""

    def __init__(self, image_size, n_actions):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        feat_size = self._get_feat_size(image_size)
        self.head = nn.Sequential(
            nn.Linear(feat_size, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions),
        )

    def _get_feat_size(self, image_size):
        dummy = torch.zeros(1, 1, image_size, image_size)
        return self.features(dummy).view(1, -1).size(1)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0).unsqueeze(0)
        elif x.dim() == 3:
            x = x.unsqueeze(1)
        features = self.features(x)
        return self.head(features.view(features.size(0), -1))


Transition = namedtuple('Transition',
                        ('state', 'action', 'reward', 'next_state', 'done'))


class ReplayBuffer:
    """Experience replay buffer for stable training."""

    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    """DQN Agent for medical image enhancement."""

    def __init__(self, image_size, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=500,
                 buffer_size=10000, batch_size=32, target_update=10):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update = target_update
        self.steps_done = 0

        self.policy_net = CNNQNetwork(image_size, n_actions).to(device)
        self.target_net = CNNQNetwork(image_size, n_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_size)

    @property
    def epsilon(self):
        return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
            np.exp(-self.steps_done / self.epsilon_decay)

    def select_action(self, state, eval_mode=False):
        if not eval_mode and random.random() < self.epsilon:
            return random.randrange(self.n_actions)

        with torch.no_grad():
            state_t = torch.FloatTensor(state).to(device)
            q_values = self.policy_net(state_t)
            return q_values.argmax(dim=1).item()

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return None

        batch = self.memory.sample(self.batch_size)

        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)

        current_q = self.policy_net(states).gather(1, actions).squeeze(1)

        with torch.no_grad():
            next_q = self.target_net(next_states).max(dim=1)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)

        loss = F.smooth_l1_loss(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        return loss.item()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())


# Instantiate
agent = DQNAgent(image_size=64, n_actions=9, lr=5e-4, gamma=0.95,
                 epsilon_decay=300, buffer_size=5000, batch_size=32)

total_params = sum(p.numel() for p in agent.policy_net.parameters())
print(f"Policy network parameters: {total_params:,}")
print(f"Architecture:\n{agent.policy_net}")

### 4.4 Training Loop

We train the DQN agent on randomly generated degraded medical images.
Episode count is kept manageable for Colab runtimes.

In [ ]:
def train_agent(agent, env, n_episodes=300, log_interval=30):
    """Training loop with medical-specific metric tracking."""
    history = {
        'episode_rewards': [], 'episode_cnr': [], 'episode_snr': [],
        'episode_ssim': [], 'episode_lengths': [], 'losses': [],
        'epsilons': [], 'action_counts': np.zeros(env.n_actions)
    }

    for episode in range(n_episodes):
        state = env.reset()
        episode_reward = 0
        done = False

        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)

            agent.memory.push(state, action, reward, next_state, float(done))
            agent.steps_done += 1

            loss = agent.train_step()
            if loss is not None:
                history['losses'].append(loss)

            episode_reward += reward
            history['action_counts'][action] += 1
            state = next_state

        if (episode + 1) % agent.target_update == 0:
            agent.update_target()

        history['episode_rewards'].append(episode_reward)
        history['episode_cnr'].append(info['cnr'])
        history['episode_snr'].append(info['snr'])
        history['episode_ssim'].append(info['ssim'])
        history['episode_lengths'].append(env.step_count)
        history['epsilons'].append(agent.epsilon)

        if (episode + 1) % log_interval == 0:
            avg_r = np.mean(history['episode_rewards'][-log_interval:])
            avg_cnr = np.mean(history['episode_cnr'][-log_interval:])
            avg_ssim = np.mean(history['episode_ssim'][-log_interval:])
            print(f"Episode {episode+1:>4d}/{n_episodes} | "
                  f"Avg Reward: {avg_r:+.4f} | "
                  f"CNR: {avg_cnr:.2f} | "
                  f"SSIM: {avg_ssim:.3f} | "
                  f"ε: {agent.epsilon:.3f}")

    return history


# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

env = MedicalImageEnv(image_size=64, max_steps=15)
agent = DQNAgent(image_size=64, n_actions=9, lr=5e-4, gamma=0.95,
                 epsilon_decay=300, buffer_size=5000, batch_size=32,
                 target_update=10)

print("Starting training...\n")
history = train_agent(agent, env, n_episodes=300, log_interval=30)
print("\n✅ Training complete!")

---

## 5. Evaluation & Results

### 5.1 Training Curves

In [ ]:
def smooth(values, window=15):
    """Exponential moving average for smoother curves."""
    smoothed = []
    last = values[0]
    weight = 2.0 / (window + 1)
    for v in values:
        last = last * (1 - weight) + v * weight
        smoothed.append(last)
    return smoothed


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Episode rewards
ax = axes[0, 0]
ax.plot(history['episode_rewards'], alpha=0.3, color='steelblue')
ax.plot(smooth(history['episode_rewards']), color='steelblue',
        linewidth=2, label='Smoothed')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Episode Reward', fontsize=11)
ax.set_title('Episode Rewards', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# CNR over episodes
ax = axes[0, 1]
ax.plot(history['episode_cnr'], alpha=0.3, color='forestgreen')
ax.plot(smooth(history['episode_cnr']), color='forestgreen',
        linewidth=2, label='Smoothed')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('CNR', fontsize=11)
ax.set_title('Contrast-to-Noise Ratio', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# SNR over episodes
ax = axes[0, 2]
ax.plot(history['episode_snr'], alpha=0.3, color='darkorange')
ax.plot(smooth(history['episode_snr']), color='darkorange',
        linewidth=2, label='Smoothed')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('SNR', fontsize=11)
ax.set_title('Signal-to-Noise Ratio', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# SSIM over episodes
ax = axes[1, 0]
ax.plot(history['episode_ssim'], alpha=0.3, color='purple')
ax.plot(smooth(history['episode_ssim']), color='purple',
        linewidth=2, label='Smoothed')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('SSIM', fontsize=11)
ax.set_title('Structural Similarity', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[1, 1]
if history['losses']:
    ax.plot(history['losses'], alpha=0.2, color='red')
    ax.plot(smooth(history['losses'], window=50), color='red',
            linewidth=2, label='Smoothed')
ax.set_xlabel('Training Step', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('DQN Training Loss', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Epsilon decay
ax = axes[1, 2]
ax.plot(history['epsilons'], color='teal', linewidth=2)
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Epsilon', fontsize=11)
ax.set_title('Exploration Rate Decay', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Training Metrics Dashboard',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Action Frequency Analysis

Which enhancement actions does the agent learn to prefer?

In [ ]:
action_counts = history['action_counts']
action_pcts = action_counts / action_counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.Set3(np.linspace(0, 1, len(env.ACTION_NAMES)))

# Bar chart
ax = axes[0]
bars = ax.barh(env.ACTION_NAMES, action_pcts, color=colors, edgecolor='gray')
ax.set_xlabel('Usage Frequency (%)', fontsize=11)
ax.set_title('Action Selection Frequency', fontsize=13, fontweight='bold')
for bar, pct in zip(bars, action_pcts):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')

# Pie chart
ax = axes[1]
nonzero = action_pcts > 1.0
labels = [n if nonzero[i] else '' for i, n in enumerate(env.ACTION_NAMES)]
ax.pie(action_pcts, labels=labels, colors=colors, autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
       startangle=90, textprops={'fontsize': 9})
ax.set_title('Action Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Learned Enhancement Strategy',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nTop 3 preferred actions:")
top3 = np.argsort(action_counts)[::-1][:3]
for rank, idx in enumerate(top3):
    print(f"  {rank+1}. {env.ACTION_NAMES[idx]} — {action_pcts[idx]:.1f}%")

### 5.3 Before/After Enhancement Comparisons

In [ ]:
def evaluate_single(agent, env, seed=None):
    """Run the trained agent on a single image and record the trajectory."""
    if seed is not None:
        np.random.seed(seed)
    state = env.reset()
    trajectory = [{'image': env.initial_image.copy(), 'action': 'initial',
                   'reward': 0}]
    done = False
    total_reward = 0

    while not done:
        action = agent.select_action(state, eval_mode=True)
        state, reward, done, info = env.step(action)
        total_reward += reward
        trajectory.append({
            'image': env.current_image.copy(),
            'action': info['action'],
            'reward': reward,
            'info': info
        })

    return trajectory, env.target.copy(), total_reward


fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for row in range(3):
    traj, target, total_r = evaluate_single(agent, env, seed=row * 17 + 7)

    degraded = traj[0]['image']
    enhanced = traj[-1]['image']

    m = MedicalImageMetrics()
    _, m_deg = m.composite_reward(degraded, target)
    _, m_enh = m.composite_reward(enhanced, target)

    axes[row, 0].imshow(target, cmap='gray')
    axes[row, 0].set_title('Target (Clean)', fontsize=11, fontweight='bold')

    axes[row, 1].imshow(degraded, cmap='gray')
    axes[row, 1].set_title(f'Degraded\nSSIM={m_deg["ssim"]:.3f}',
                           fontsize=11, fontweight='bold')

    axes[row, 2].imshow(enhanced, cmap='gray')
    axes[row, 2].set_title(f'RL Enhanced\nSSIM={m_enh["ssim"]:.3f}',
                           fontsize=11, fontweight='bold')

    diff = np.abs(enhanced - degraded)
    axes[row, 3].imshow(diff, cmap='hot')
    axes[row, 3].set_title('Change Map', fontsize=11, fontweight='bold')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Before/After Enhancement Comparisons',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.4 Enhancement Sequence Visualization

Step-by-step view of the agent's decision-making on a single image.

In [ ]:
traj, target, total_r = evaluate_single(agent, env, seed=99)

n_steps = min(len(traj), 8)  # show up to 8 steps
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i in range(n_steps):
    ax = axes[i // 4, i % 4]
    ax.imshow(traj[i]['image'], cmap='gray')
    action_label = traj[i]['action']
    r_val = traj[i]['reward']
    if i == 0:
        ax.set_title('Step 0: Input', fontsize=11, fontweight='bold')
    else:
        ax.set_title(f'Step {i}: {action_label}\nr={r_val:+.3f}',
                     fontsize=10, fontweight='bold')
    ax.axis('off')

# Fill remaining subplots if fewer than 8 steps
for i in range(n_steps, 8):
    axes[i // 4, i % 4].axis('off')

plt.suptitle('Enhancement Sequence — Agent\'s Step-by-Step Decisions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nAction sequence:")
for i, step in enumerate(traj):
    if i == 0:
        print("  [Start] Degraded input")
    else:
        print(f"  Step {i}: {step['action']:>20s}  (reward: {step['reward']:+.4f})")
print(f"\nTotal reward: {total_r:+.4f}")

### 5.5 Comparison with Fixed Enhancement Pipeline

Does the RL agent outperform a hand-crafted enhancement sequence?

In [ ]:
def fixed_pipeline(image):
    """A hand-crafted 'radiologist-designed' enhancement pipeline."""
    result = image.copy()

    # Step 1: Denoise with Gaussian
    result = ndimage.gaussian_filter(result, sigma=0.8)

    # Step 2: Histogram equalization
    hist, bins = np.histogram(result.flatten(), bins=256,
                              range=(0, 1), density=True)
    cdf = hist.cumsum()
    cdf = cdf / cdf[-1]
    result = np.interp(result.flatten(), bins[:-1], cdf).reshape(result.shape)

    # Step 3: Contrast adjustment
    mean = result.mean()
    result = mean + 1.3 * (result - mean)

    # Step 4: Sharpen
    blurred = ndimage.gaussian_filter(result, sigma=1.0)
    result = result + 0.5 * (result - blurred)

    return np.clip(result, 0, 1)


n_eval = 50
rl_scores = {'ssim': [], 'cnr': [], 'snr': [], 'reward': []}
fixed_scores = {'ssim': [], 'cnr': [], 'snr': [], 'reward': []}
degraded_scores = {'ssim': [], 'cnr': [], 'snr': [], 'reward': []}
m = MedicalImageMetrics()

for i in range(n_eval):
    np.random.seed(i + 1000)
    state = env.reset()
    target = env.target.copy()
    degraded = env.initial_image.copy()

    # RL agent
    done = False
    while not done:
        action = agent.select_action(state, eval_mode=True)
        state, _, done, _ = env.step(action)
    rl_enhanced = env.current_image.copy()

    # Fixed pipeline
    fixed_enhanced = fixed_pipeline(degraded)

    r_rl, m_rl = m.composite_reward(rl_enhanced, target)
    r_fix, m_fix = m.composite_reward(fixed_enhanced, target)
    r_deg, m_deg = m.composite_reward(degraded, target)

    for key in ['ssim', 'cnr', 'snr']:
        rl_scores[key].append(m_rl[key])
        fixed_scores[key].append(m_fix[key])
        degraded_scores[key].append(m_deg[key])
    rl_scores['reward'].append(r_rl)
    fixed_scores['reward'].append(r_fix)
    degraded_scores['reward'].append(r_deg)


# Comparison plot
metrics_to_plot = ['ssim', 'cnr', 'snr', 'reward']
labels = ['SSIM', 'CNR', 'SNR', 'Composite Reward']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, (metric, label) in enumerate(zip(metrics_to_plot, labels)):
    ax = axes[i]
    data = [degraded_scores[metric], fixed_scores[metric], rl_scores[metric]]
    bp = ax.boxplot(data, labels=['Degraded', 'Fixed', 'RL Agent'],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    colors_box = ['#ff9999', '#99ccff', '#99ff99']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'RL Agent vs Fixed Pipeline (n={n_eval} images)',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("\nAverage Metrics Comparison:")
print(f"{'Metric':<12} {'Degraded':>10} {'Fixed':>10} {'RL Agent':>10}")
print("-" * 44)
for metric, label in zip(metrics_to_plot, labels):
    d_val = np.mean(degraded_scores[metric])
    f_val = np.mean(fixed_scores[metric])
    r_val = np.mean(rl_scores[metric])
    print(f"{label:<12} {d_val:>10.4f} {f_val:>10.4f} {r_val:>10.4f}")

### 5.6 Side-by-Side Visual Comparison

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
col_titles = ['Target', 'Degraded', 'Fixed Pipeline', 'RL Agent']

for row in range(3):
    np.random.seed(row + 2000)
    state = env.reset()
    target = env.target.copy()
    degraded = env.initial_image.copy()

    # RL
    done = False
    while not done:
        action = agent.select_action(state, eval_mode=True)
        state, _, done, _ = env.step(action)
    rl_img = env.current_image.copy()

    # Fixed
    fixed_img = fixed_pipeline(degraded)

    images = [target, degraded, fixed_img, rl_img]
    for col, img in enumerate(images):
        ax = axes[row, col]
        ax.imshow(img, cmap='gray')
        if row == 0:
            ax.set_title(col_titles[col], fontsize=13, fontweight='bold')

        if col > 0:
            s = ssim(img, target, data_range=target.max() - target.min())
            ax.set_xlabel(f'SSIM: {s:.3f}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Visual Comparison: Fixed Pipeline vs RL Agent',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## 6. Clinical Relevance Discussion

### 6.1 Translation to Real Medical Imaging

The framework demonstrated here translates directly to clinical systems:

| Aspect | This Demo | Real Deployment |
|--------|-----------|------------------|
| Images | 64×64 synthetic | 512–2048px DICOM |
| Actions | 9 basic filters | Vendor-specific processing, learned filters |
| Reward | SSIM + CNR + SNR | Radiologist ratings, downstream task accuracy |
| Training | 300 episodes | Thousands, on curated datasets |
| State | Single grayscale | Multi-window, 3D volumes |

Key advantages of the RL approach over fixed pipelines:

1. **Adaptive**: Different enhancement sequences for different image characteristics
2. **Optimizable**: Reward function can incorporate task-specific objectives
3. **Generalizable**: Same architecture works across modalities with re-training

### 6.2 Regulatory Considerations

Medical image processing software is regulated by the **FDA (510(k), De Novo)** and
**EU MDR**. Key considerations:

- **Intended use**: Enhancement for *display* vs *diagnosis* has different regulatory pathways
- **Determinism**: RL policies are deterministic at inference (greedy), satisfying reproducibility requirements
- **Validation**: Must demonstrate non-inferiority vs predicate devices on clinical endpoints
- **Transparency**: Action sequences provide an audit trail (which operations were applied)
- **Failure modes**: The `no_op` action allows the system to "do no harm" when uncertain

### 6.3 Limitations

1. **Synthetic data**: Real medical images have far more complex anatomy and pathology
2. **Action granularity**: Fixed-strength actions; continuous action spaces would be more flexible
3. **Single-scale**: Real systems need multi-scale, multi-window processing
4. **No pathology awareness**: Enhancement should preserve (not obscure) pathological findings
5. **Reward engineering**: Clinical reward functions require expert radiologist input

### 6.4 Future Directions

- **Continuous action spaces** with actor-critic methods (SAC, PPO)
- **Multi-agent** systems for 3D volume processing (slice-by-slice coordination)
- **Human-in-the-loop RL**: Use radiologist feedback as reward signal
- **Pathology-preserving constraints**: Ensure no diagnostic information is lost
- **Foundation model features**: Use pre-trained medical vision encoders as state representations

---

## 7. Summary

In this module, we built a **complete RL system for medical image enhancement**:

### What We Built

| Component | Implementation |
|-----------|----------------|
| **Data** | Synthetic X-ray and CT image generator with configurable degradations |
| **Environment** | Gym-like MDP with 9 enhancement actions |
| **Metrics** | CNR, SNR, SSIM, artifact penalty — clinically motivated |
| **Agent** | DQN with CNN feature extractor, experience replay, target network |
| **Evaluation** | Training curves, action analysis, before/after comparisons |

### Key Mathematical Concepts

- **MDP formulation**: $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$ for sequential enhancement
- **Composite reward**: $R = w_1 \cdot \text{SSIM} + w_2 \cdot \text{CNR} + w_3 \cdot \text{SNR} - w_4 \cdot \text{artifacts}$
- **DQN loss**: $\mathcal{L} = \mathbb{E}\left[L_\delta(r + \gamma \max_{a'} Q(s',a';\theta^-) - Q(s,a;\theta))\right]$
- **Clinical metrics**: CNR for tissue contrast, SNR for noise assessment

### Key Takeaways

1. RL enables **adaptive** enhancement that adjusts to each image
2. **Medical-specific reward functions** guide the agent toward clinically useful results
3. The agent learns meaningful strategies (preferring certain actions over others)
4. **Regulatory and clinical** considerations are essential for real deployment
5. The framework is extensible to continuous actions, 3D volumes, and human feedback

---

*This notebook is part of the "Reinforcement Learning for Image Processing" course.*
*Module 12.1 — Real-World Projects: Medical Image Enhancement*